# Spellchecker Quest

Someone introduced mistakes into Taras Shevchenko's poems. Our task is to fix these mistakes and uncover the hidden message.

## Task

You will receive training and testing data.

The training data is available in the `lab.train_text` field. This is plain unannotated text that you need to use to train a language model. Any model will work. I recommend a feed-forward neural model with character-level tokenization, as we covered in the last lecture. However, an n-gram model should also work.

The testing data is in the `lab.test_items` field. Here's an example of a test item:

```json
{
  "text": "They sang prose, but on the knives,",
  "error_start": 20,
  "error_end": 25,
  "error": "knives",
  "corrections": [
    "feet",
    "knots",
    "notes",
    "knives",
    "votes"
  ]
}
```

`error_start` and `error_end` indicate the position of the error in the text (by character index). In this example, the error is `text[20:25]`, i.e., the word "knives."

`corrections` is a list of possible fixes.

Your task is to select the correct fix from the list.

## Hidden Message

Each test item in `lab.test_items` reveals one character of a hidden message. To extract it, find the difference between the letters in the error word (`error`) and the selected correction. Print this letter. If the error word is actually correct, which sometimes happens, print a space. Examples:

```
Error               Correction     To print
-------------------------------------------
hello               hullo          u
cat                 cart           r
ballet              balet          l
hello               hello          (space)
```

The hidden message you uncover is a line from a poem by a Ukrainian author.

The answer to the quest is the author's name in the format "First Last."

Let's go! 🚀

In [ ]:
!pip install --quiet --ignore-installed http://nlp.band/static/pypy/lpnlp-2023.10.2-py3-none-any.whl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.3/167.3 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.8/144.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.3/126.3 kB 5.1 MB/s eta 0:00:00


In [ ]:
import lpnlp

lab = lpnlp.start(
    email="sofiia.tkach.kn.2021@lpnu.ua",             # <----------- Заповніть це поле
    lab="quest_spellchecker"
    )

Удачі!


##Language Model
Train your language model here.

In [ ]:
print(lab.train_text[:330])

﻿ПРИЧИННА

Реве та стогне Дніпр широкий,
Сердитий вітер завива,
Додолу верби гне високі,
Горами хвилю підійма.
І блідий місяць на ту пору
Із хмари де-де виглядав,
Неначе човен в синім морі,
То виринав, то потопав.
Ще треті півні не співали,
Ніхто нігде не гомонів,
Сичі в гаю перекликались,
Та ясен раз у раз скрипів.


In [ ]:
from collections import Counter

class NgramModel:
    def __init__(self, vocab: list[str], n: int):
        # Model Initialization
        # vocab: a dictionary of tokens used in the model
        # n: the number of tokens in the n-gram
        self.n = n
        self.vocab = vocab
        self.ngram_counts = Counter()# Словник для підрахунку частоти n-гр

    def train(self, tokens: list[str]) -> None:
        # Function for training the model based on input tokens
        tokens = [t for t in tokens if t in self.vocab]  # Фільтруємо токени, щоб залишити лише ті, які є в словнику
        # Generate n-grams and count their frequencies
        self.ngram_counts = Counter(self.generate_ngrams(tokens, self.n))

    def generate_ngrams(self, tokens: list[str], n: int) -> list[tuple]:
        # Generates n-grams from the provided list of tokens
        # Returns a list of n-grams (tuples) containing n tokens
        return list(zip(*[tokens[i:] for i in range(n)]))


In [ ]:
def main(n: int, vocab_size: int):
    # n: number of tokens in the n-gram
    # vocab_size: maximum number of words in the vocabulary
    vocab, model = train(n, vocab_size)
    return vocab, model

def train(n: int, vocab_size: int) -> tuple[list[str], NgramModel]:
    # Trains the model and returns the vocabulary and the model

    # Get training text and split it into lines
    lines = lab.train_text.splitlines()
    tokens = []

    # Get training text and split it into lines
    for line in lines:
        tokens += ["<s>"] + tokenize(line) + ["</s>"]# Додаємо початкові та кінцеві теги <s> і </s> для контексту

    # Count the frequency of each token
    counts = Counter(tokens)

    # Select the top vocab_size words for the vocabulary based on frequency
    vocab = [token for token, count in counts.most_common(vocab_size)]

    # Create and train the model
    model = NgramModel(vocab, n)
    model.train(tokens=tokens)

    # Return the vocabulary and the model
    return vocab, model

def tokenize(text: str) -> list[str]:
    """Splits text into tokens (words)."""
    return text.split()

if __name__ == '__main__':
    vocab, model = main(5, 22522)

    print(f"Vocabulary size: {len(vocab)}")
    print(f"Top 10 words: {vocab[:10]}")
    print(f"N-gram counts: {list(model.ngram_counts.items())[:10]}")


22522
Vocabulary size: 22522
Top 10 words: ['<s>', '</s>', 'І', 'не', 'А', 'в', 'на', 'і', 'Та', 'й']
N-gram counts: [(('<s>', '\ufeffПРИЧИННА', '</s>', '<s>', '</s>'), 1), (('\ufeffПРИЧИННА', '</s>', '<s>', '</s>', '<s>'), 1), (('</s>', '<s>', '</s>', '<s>', 'Реве'), 1), (('<s>', '</s>', '<s>', 'Реве', 'та'), 1), (('</s>', '<s>', 'Реве', 'та', 'стогне'), 1), (('<s>', 'Реве', 'та', 'стогне', 'Дніпр'), 1), (('Реве', 'та', 'стогне', 'Дніпр', 'широкий,'), 1), (('та', 'стогне', 'Дніпр', 'широкий,', '</s>'), 1), (('стогне', 'Дніпр', 'широкий,', '</s>', '<s>'), 1), (('Дніпр', 'широкий,', '</s>', '<s>', 'Сердитий'), 1)]


# Читаємо між рядків

In [ ]:
import math
import collections
from typing import List, Tuple


# Допоміжна фунція:
def get_letter(w1: str, w2: str) -> str:
    """Returns the letter by which the words differ or a space if the words are identical."""

    letters1 = collections.Counter(w1)
    letters2 = collections.Counter(w2)

    diff = letters1 - letters2
    if len(diff) != 1:
        return " "

    return diff.most_common()[0][0]


def score_text(text: str, model, vocab) -> float:
    # Function to evaluate the logarithmic probability of a text based on n-grams
    # text: the text string for which the probability needs to be calculated
    # model: the n-gram model object
    # vocab: the vocabulary of tokens used in the model

    # Split the text into tokens and filter them based on the vocabulary
    tokens = [token for token in text.split() if token in vocab]

    n = model.n  # Get the value of n for n-grams
    total_ngrams = sum(model.ngram_counts.values())  # Total number of n-grams in the model
    log_prob = 0  # Initialize the logarithmic probability
    alpha = 0.5  # Smoothing parameter

    # Iterate over the n-grams, starting from n and decreasing to 1 (backoff)
    for i in range(n, 0, -1):
        # Generate n-grams from the tokens for the current value of i
        ngrams = model.generate_ngrams(tokens, i)

         # For each generated n-gram
        for ngram in ngrams:
            # Get the count of this n-gram from the model
            ngram_count = model.ngram_counts.get(ngram, 0)

            vocab_size = len(vocab) # Get the size of the vocabulary

            # Calculate the probability of the n-gram using Laplace smoothing
            ngram_prob = (ngram_count + alpha) / (total_ngrams + alpha * vocab_size)

            # Add the logarithm of the probability to the total logarithmic probability
            log_prob += math.log(ngram_prob)

    return log_prob  # Return the total logarithmic probability



def solve(model, vocab, test_items) -> Tuple[List[str], str]:
    """Returns a list of corrected words for each text in test_items and
    the secret message.
    """

    choices = []
    secret = []

    for item in test_items:
        scores = []
        for corr in item['corrections']:

            # Replace the error word in the text with the candidate word
            text = item['text'][:item['error_start']] + corr + item['text'][item['error_end']:]

            # Calculate the score of the text
            score = score_text(text, model, vocab)
            scores.append(score)

            # print(f'{score:.4f} {text}')

        # Sort candidates by their score
        result = sorted(zip(scores, item['corrections']), key=lambda x: x[0])

        # Choose the best replacement
        best = result[0]
        best_word = best[1]
        choices.append(best_word)

        # Find the next letter of the message
        error = item['error']
        letter = get_letter(error, best_word)
        secret.append(letter)

    secret_message = ''.join(secret)

    return choices, secret_message

choices, secret_message = solve(model, vocab, lab.test_items)

lab.evaluate_accuracy(choices)
print("SECRET MESSAGE: ", secret_message)


Відповідь правильна ✅
accuracy = 0.64. Непогано! Спробуй розгадай приховане повідомлення
SECRET MESSAGE:  д ми м ї  у и м їулихонм ні з В ми на о  Талим а папер  с мним  рядамитчом   с вітерч е р зв  в в ст пууя  Пил нуечом васм  х  не пв сПалобяк Свою дит ну за карі  очен т  затчорн ї б о и  ер е рв л ся с   ло ь вил  ало мов  ви  вало як умі о за темнії  очі з дв  н  и  са т е  н й з нла ки діво ін а стеПи та за м гил тщ лн   країні се це млі о н ц отілошСпів  ь  а чуж ні


In [ ]:
lab.answer("Тарас Шевченко")

Відповідь правильна ✅
Правильно! 🚀 Заповни тепер цю форму, будь ласка: https://tally.so/r/wkl0zZ


correct answer

Відправте посилання на цей colab або PDF з ним на пошту oleksii.o.syvokon@lpnu.ua. Дякую!
